# 📓 Update dataset with last boxscores and matches

# Please first run full pipeline until merge_clean_seasons_boxscores to have base data to work with and merge

In [1]:

import os
import pandas as pd
from datetime import datetime
from src.config import *
from src.utils import *
from src.nba_scrapping import *



In [2]:
# ⚙️ Initialisation du run
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
current_season = '2024-25'  # À rendre dynamique si besoin plus tard


In [3]:

# 📁 Préparation des dossiers de sortie
os.makedirs(DATA_LAST_GAMES_DIR, exist_ok=True)
os.makedirs(DATA_LAST_BOXSCORES_BATCHES_DIR, exist_ok=True)

# 🔄 Chargement des historiques si existants
hist_games_path = get_latest_file(DATA_LAST_GAMES_MERGED_DIR)


In [4]:

# 📥 1. Téléchargement des matchs de la saison actuelle
print("\n📥 Téléchargement des nouveaux matchs pour la saison:", current_season)
matchs_output_dir = os.path.join(DATA_LAST_GAMES_DIR, current_season)
last_games_path = download_games_for_seasons([current_season], matchs_output_dir, run_timestamp, max_retries=10)


📥 Téléchargement des nouveaux matchs pour la saison: 2024-25
Extraction saison 2024-25


In [5]:

# 📊 2. Comparaison avec les données historiques pour trouver les nouveaux matchs
games_to_scrape = get_new_games(hist_games_path, last_games_path)
print(f"✅ {len(games_to_scrape)} nouveaux matchs trouvés à scraper.")

if games_to_scrape.empty:
    print("✅ Aucun nouveau match à scraper. Fin du script.")
    exit(0)

2 nouveaux matchs à traiter
✅ 2 nouveaux matchs trouvés à scraper.


In [6]:
games_to_scrape

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON
2626,42024,1610612754,IND,Indiana Pacers,0042400401,2025-06-05,IND @ OKC,W,240,111,...,13,43,56,24,1,7,24,22,1.0,2024-25
2627,42024,1610612760,OKC,Oklahoma City Thunder,0042400401,2025-06-05,OKC vs. IND,L,239,110,...,10,29,39,13,14,6,6,19,-1.0,2024-25


In [7]:

# 💾 3. Merge historique + nouveaux matchs
historical_games = pd.read_csv(hist_games_path, low_memory=False, dtype={'GAME_ID': str})
all_games = pd.concat([historical_games, games_to_scrape], ignore_index=True)

save_dataframe_to_csv(all_games, DATA_LAST_GAMES_MERGED_DIR, 'games_merged_all_seasons_', run_timestamp)
print("✅ Jeux de matchs fusionnés sauvegardés.")


✅ Jeux de matchs fusionnés sauvegardés.


In [8]:

# 🏀 4. Scraping des nouveaux boxscores
print(f"--- Traitement de la saison {current_season} ---")
season_df = games_to_scrape[games_to_scrape['SEASON'] == current_season]
season_output_dir = os.path.join(DATA_LAST_BOXSCORES_BATCHES_DIR, run_timestamp, current_season)
scrape_boxscores_v3_for_games(season_df, season_output_dir,max_retries=10)

--- Traitement de la saison 2024-25 ---
[DEBUG] Found 0 batch files for endpoint 'traditional' in season 2024-25
[DEBUG] Found 0 batch files for endpoint 'advanced' in season 2024-25
[DEBUG] Found 0 batch files for endpoint 'fourfactors' in season 2024-25
[DEBUG] Found 0 batch files for endpoint 'misc' in season 2024-25
[DEBUG] Found 0 batch files for endpoint 'scoring' in season 2024-25
[DEBUG] Found 0 batch files for endpoint 'usage' in season 2024-25
--------- 1 GAME_ID to scrap for season 2024-25 ---------
[1/1] GAME_ID: 0042400401 - 2025-06-05
 traditional V3 batch 001 saved (29 rows)
 advanced V3 batch 001 saved (29 rows)
 fourfactors V3 batch 001 saved (29 rows)
 misc V3 batch 001 saved (29 rows)
 scoring V3 batch 001 saved (29 rows)
 usage V3 batch 001 saved (29 rows)


True

In [9]:
print("\n✅ Mise à jour complète terminée. Données prêtes pour le feature engineering.")



✅ Mise à jour complète terminée. Données prêtes pour le feature engineering.
